In [1]:
import os, json, random, time
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

SEED = 42
DATA_DIR = "processed_data"
TEXT_COL, LABEL_COL = "instruction_clean", "intent"

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
val   = pd.read_csv(f"{DATA_DIR}/val.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

print("Train / Val / Test:", len(train), "/", len(val), "/", len(test))
print("Number of intents:", train[LABEL_COL].nunique(), "| Number of categories:", train["category"].nunique())

dist = pd.DataFrame({
    "train": train[LABEL_COL].value_counts(normalize=True),
    "val":   val[LABEL_COL].value_counts(normalize=True),
    "test":  test[LABEL_COL].value_counts(normalize=True),
}).mul(100).round(2)
dist["max_diff"] = dist.max(axis=1) - dist.min(axis=1)
print("\nLargest difference in any intent's share across the splits (percentage points):", dist["max_diff"].max().round(2))
dist.sort_values("max_diff", ascending=False).head(5)

Train / Val / Test: 19419 / 2427 / 2428
Number of intents: 27 | Number of categories: 11

Largest difference in any intent's share across the splits (percentage points): 0.05


,train,val,test,max_diff
intent,,,,
track_order,3.22,3.26,3.21,0.05
change_order,3.53,3.54,3.50,0.04
cancel_order,1.80,1.81,1.77,0.04
create_account,3.60,3.58,3.62,0.04
check_payment_methods,4.09,4.08,4.12,0.04


In [3]:
tr_set, va_set, te_set = set(train[TEXT_COL]), set(val[TEXT_COL]), set(test[TEXT_COL])
print("overlap train∩val :", len(tr_set & va_set))
print("overlap train∩test:", len(tr_set & te_set))
print("overlap val∩test  :", len(va_set & te_set))

print("Intents missing from val :", set(train[LABEL_COL]) - set(val[LABEL_COL]))
print("Intents missing from test:", set(train[LABEL_COL]) - set(test[LABEL_COL]))

overlap train∩val : 0
overlap train∩test: 0
overlap val∩test  : 0
Intents missing from val : set()
Intents missing from test: set()


In [4]:
RAW = "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
if os.path.exists(RAW) and os.path.exists("preprocessing.py"):
    from preprocessing import clean_text
    from sklearn.model_selection import train_test_split

    raw = pd.read_csv(RAW)
    raw[TEXT_COL] = raw["instruction"].apply(clean_text)
    dd = raw.drop_duplicates(subset=TEXT_COL, keep="first").reset_index(drop=True)

    tr2, tmp = train_test_split(dd, test_size=0.2, stratify=dd[LABEL_COL], random_state=SEED)
    va2, te2 = train_test_split(tmp, test_size=0.5, stratify=tmp[LABEL_COL], random_state=SEED)

    same = lambda a, b: len(a) == len(b) and (a[TEXT_COL].values == b[TEXT_COL].values).all()
    print("train matches:", same(tr2, train), "| val matches:", same(va2, val), "| test matches:", same(te2, test))
else:
    print("Raw data not found next to the notebook; continuing with the prepared split files.")

train matches: True | val matches: True | test matches: True


In [5]:
def make_tfidf():
    return FeatureUnion([
        ("word", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2)),
        ("char", TfidfVectorizer(sublinear_tf=True, analyzer="char_wb", ngram_range=(2, 5), min_df=2)),
    ])

X_train, y_train = train[TEXT_COL], train[LABEL_COL]
X_val,   y_val   = val[TEXT_COL],   val[LABEL_COL]
X_test,  y_test  = test[TEXT_COL],  test[LABEL_COL]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def run_grid(name, clf, grid):
    pipe = Pipeline([("tfidf", make_tfidf()), ("clf", clf)])
    gs = GridSearchCV(pipe, grid, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True, return_train_score=False)
    t0 = time.time()
    gs.fit(X_train, y_train)
    print(f"[{name}] best CV Macro-F1 = {gs.best_score_:.4f} | best params = {gs.best_params_} | {time.time()-t0:.0f}s")
    return gs

def top_results(gs, k=6):
    r = pd.DataFrame(gs.cv_results_)
    r = r.sort_values(["mean_test_score", "std_test_score"], ascending=[False, True])
    return r[["params", "mean_test_score", "std_test_score", "mean_fit_time"]].head(k).round(4)

In [6]:
svc_grid = {
    "clf__C": [0.1, 0.3, 1, 3, 10],
    "clf__class_weight": [None, "balanced"],
}
gs_svc = run_grid("LinearSVC", LinearSVC(max_iter=10000, random_state=SEED), svc_grid)
top_results(gs_svc)

[LinearSVC] best CV Macro-F1 = 0.9991 | best params = {'clf__C': 10, 'clf__class_weight': 'balanced'} | 235s


,params,mean_test_score,std_test_score,mean_fit_time
9,"{'clf__C': 10, 'clf__class_weight': 'balanced'}",0.9991,0.0003,5.1416
8,"{'clf__C': 10, 'clf__class_weight': None}",0.9990,0.0003,5.0417
7,"{'clf__C': 3, 'clf__class_weight': 'balanced'}",0.9990,0.0004,4.2741
6,"{'clf__C': 3, 'clf__class_weight': None}",0.9989,0.0005,4.1399
5,"{'clf__C': 1, 'clf__class_weight': 'balanced'}",0.9987,0.0005,4.0622
4,"{'clf__C': 1, 'clf__class_weight': None}",0.9986,0.0005,3.9911


**📌 Result (LinearSVC):** the best setting is `C=10` + `class_weight="balanced"` with CV Macro-F1 = **0.9991** (±0.0003). Among the top settings the differences are tiny (0.9986 to 0.9991), and `balanced` is slightly ahead of `None` at each of the top `C` values.

⚠️ `C=10` is the **largest value** in the grid, so the best point sits on the edge. We could add 30 and 100 to check, but the expected difference is small (within the std), so it probably would not change the conclusion.

In [7]:
lr_grid = {
    "clf__C": [1, 10, 50, 100],
    "clf__class_weight": [None, "balanced"],
}
gs_lr = run_grid("LogisticRegression", LogisticRegression(max_iter=2000, random_state=SEED), lr_grid)
top_results(gs_lr)

[LogisticRegression] best CV Macro-F1 = 0.9986 | best params = {'clf__C': 100, 'clf__class_weight': 'balanced'} | 143s


,params,mean_test_score,std_test_score,mean_fit_time
7,"{'clf__C': 100, 'clf__class_weight': 'balanced'}",0.9986,0.0004,2.8875
5,"{'clf__C': 50, 'clf__class_weight': 'balanced'}",0.9984,0.0006,2.9491
4,"{'clf__C': 50, 'clf__class_weight': None}",0.9984,0.0009,3.1870
2,"{'clf__C': 10, 'clf__class_weight': None}",0.9982,0.0004,3.2754
6,"{'clf__C': 100, 'clf__class_weight': None}",0.9982,0.0007,3.1397
3,"{'clf__C': 10, 'clf__class_weight': 'balanced'}",0.9981,0.0004,3.5391


**📌 Result (LogisticRegression):** the best setting is `C=100` + `balanced` with CV Macro-F1 = **0.9986**. Again the best `C` is on the edge of the grid, and the model prefers very weak regularization (the data is easy to separate). It is slightly below LinearSVC in CV.

In [8]:
models = {"LinearSVC": gs_svc, "LogisticRegression": gs_lr}

rows = []
for name, gs in models.items():
    pred = gs.predict(X_val)
    rows.append({
        "model": name,
        "cv_macro_f1": round(gs.best_score_, 4),
        "val_macro_f1": round(f1_score(y_val, pred, average="macro"), 4),
        "val_accuracy": round(accuracy_score(y_val, pred), 4),
        "val_errors": int((pred != y_val).sum()),
        "best_params": gs.best_params_,
    })
val_table = pd.DataFrame(rows).set_index("model")
val_table

,cv_macro_f1,val_macro_f1,val_accuracy,val_errors,best_params
model,,,,,
LinearSVC,0.9991,0.9990,0.9992,2,"{'clf__C': 10, 'clf__class_weight': 'balanced'}"
LogisticRegression,0.9986,0.9994,0.9996,1,"{'clf__C': 100, 'clf__class_weight': 'balanced'}"


**📌 Comparison on val:** the difference between the two models is **one mistake** (LinearSVC: 2 errors, LogisticRegression: 1) out of 2427 examples, so it is not a real difference. That is why the selection uses the CV, which relies on about 19K examples × 5 folds and comes with a std, and it gives a slight edge to **LinearSVC**.

In [9]:
final_name = max(models, key=lambda n: models[n].best_score_)
print("Selected model (highest CV Macro-F1):", final_name)

test_rows, test_preds = [], {}
for name, gs in models.items():
    p = gs.predict(X_test)
    test_preds[name] = p
    test_rows.append({
        "model": name,
        "test_macro_f1": round(f1_score(y_test, p, average="macro"), 4),
        "test_accuracy": round(accuracy_score(y_test, p), 4),
        "test_errors": int((p != y_test).sum()),
    })
test_table = pd.DataFrame(test_rows).set_index("model")
test_table

Selected model (highest CV Macro-F1): LinearSVC


,test_macro_f1,test_accuracy,test_errors
model,,,
LinearSVC,0.9992,0.9992,2
LogisticRegression,0.9988,0.9988,3


In [10]:
pred = test_preds[final_name]
rep = pd.DataFrame(classification_report(y_test, pred, output_dict=True, zero_division=0)).T
rep = rep.drop(["accuracy", "macro avg", "weighted avg"]).sort_values("f1-score")
print("5 weakest classes by F1:")
display(rep.head(5).round(4))

err = test.assign(pred=pred)[pred != y_test]
print(f"\nNumber of errors: {len(err)} out of {len(test)}")
if len(err):
    display(err.groupby([LABEL_COL, "pred"]).size().sort_values(ascending=False).head(10).rename("count").reset_index())
    display(err[["instruction", LABEL_COL, "pred", "flags"]].head(10))

5 weakest classes by F1:


,precision,recall,f1-score,support
change_order,1.0000,0.9882,0.9941,85.0
get_invoice,1.0000,0.9888,0.9944,89.0
delete_account,0.9890,1.0000,0.9945,90.0
check_invoice,0.9892,1.0000,0.9946,92.0
check_cancellation_fee,1.0000,1.0000,1.0000,94.0



Number of errors: 2 out of 2428


,intent,pred,count
0,change_order,delete_account,1
1,get_invoice,check_invoice,1


,instruction,intent,pred,flags
587,i nmeed assistance removing some products,change_order,delete_account,BLMQZ
1181,gedtting bill from {{Person Name}},get_invoice,check_invoice,BKLZ


**📌 Error analysis:** the selected model made **only 2 mistakes out of 2428**, both on sentences with typos (flag `Z`):
- `"i nmeed assistance removing some products"` (correct: `change_order`, model said `delete_account`): the sentence itself is genuinely ambiguous between modifying an order and deleting something.
- `"gedtting bill from ..."` (correct: `get_invoice`, model said `check_invoice`): the two classes are close in meaning.

So the remaining errors reflect label ambiguity and typos more than a weakness of the model.

In [11]:
def add_typos(text, p, rng):
    out = []
    for w in text.split():
        if w.startswith("ph_") or len(w) < 4 or rng.random() > p:
            out.append(w)
            continue
        i = rng.randrange(1, len(w) - 1)
        op = rng.choice(["swap", "drop", "dup", "sub"])
        if op == "swap":
            w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
        elif op == "drop":
            w = w[:i] + w[i + 1:]
        elif op == "dup":
            w = w[:i] + w[i] + w[i:]
        else:
            w = w[:i] + rng.choice("abcdefghijklmnopqrstuvwxyz") + w[i + 1:]
        out.append(w)
    return " ".join(out)

levels, seeds = [0.0, 0.1, 0.3, 0.5], [1, 2, 3]
rob_rows = []
for name, gs in models.items():
    for p in levels:
        scores = []
        for s in (seeds if p > 0 else [0]):
            rng = random.Random(s)
            noisy = X_val.apply(lambda t: add_typos(t, p, rng))
            scores.append(f1_score(y_val, gs.predict(noisy), average="macro"))
        rob_rows.append({"model": name, "typo_rate": p, "macro_f1": round(float(np.mean(scores)), 4)})

robust = pd.DataFrame(rob_rows).pivot(index="typo_rate", columns="model", values="macro_f1")
robust

model,LinearSVC,LogisticRegression
typo_rate,,
0.0,0.9990,0.9994
0.1,0.9978,0.9979
0.3,0.9932,0.9896
0.5,0.9849,0.9812


**📌 Robustness:** without typos the two models are close (0.9990 vs 0.9994), but as typos increase **LinearSVC holds up better**: at 30% typos (0.9932 vs 0.9896) and at 50% (0.9849 vs 0.9812). This is an additional reason for choosing it.

 Saving the model and the results


In [12]:
os.makedirs("artifacts", exist_ok=True)
for name, gs in models.items():
    joblib.dump(gs.best_estimator_, f"artifacts/{name.lower()}_pipeline.joblib")

results = {
    "final_model": final_name,
    "val": val_table.drop(columns="best_params").to_dict(orient="index"),
    "test": test_table.to_dict(orient="index"),
    "robustness_val_macro_f1": {str(k): v for k, v in robust.to_dict(orient="index").items()},
    "best_params": {n: {k: (None if v is None else v) for k, v in gs.best_params_.items()} for n, gs in models.items()},
}
with open("artifacts/results_classical.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

clf = joblib.load(f"artifacts/{final_name.lower()}_pipeline.joblib")
print(clf.predict(["i need help cancelling puchase ph_order_number"]))

['cancel_order']
